In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path


def locate_bundle() -> Path:
    starts = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in starts:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "experiments").is_dir() and (candidate / "scripts").is_dir()),
    BUNDLE_DIR,
)
sys.path.append(str(PROJECT_DIR))

# Required for live tile loading. Leave empty for the included six-tile examples.
COUNTING_DATA_ROOT = os.environ.get("COUNTING_DATA_ROOT", "")
TASKS = [
    "malaria/trophozoite", "malaria/difficult", "malaria/red_blood_cell",
    "birds/bird", "dota/tennis_court", "dota/harbor", "dota/ship",
    "dota/plane", "dota/small_vehicle", "kenyan_wildlife/giraffe",
    "dota/helicopter", "dota/storage_tank",
]
MODEL_LABEL = "Qwen3-VL-8B-Instruct"
BATCH_SIZE = 8
MAX_CHOICE = 300
GPU_MEMORY_UTILIZATION = 0.85
LABEL_ROOT = PROJECT_DIR / "outputs" / "mllm_all_tiles_all_models"
PROPRIETARY_ROOT = PROJECT_DIR / "outputs" / "proprietary_all_tiles_260710"
RUN_ROOT = PROJECT_DIR / "outputs" / "runs"
EXAMPLE_CSV = BUNDLE_DIR / "data" / "example_tiles.csv"
OPEN_SOURCE_CSV = PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "open_source_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_CSV = PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv" if (PROJECT_DIR / "outputs" / "runs" / "proprietary_all_tiles_current.csv").is_file() else BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
OPENROUTER_KEY_SELECTOR = "OPENROUTER_API_KEY"

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

PREDICTION_CSVS = [PROJECT_DIR / "outputs/runs/open_source_all_tiles_current.csv", PROJECT_DIR / "outputs/runs/proprietary_all_tiles_current.csv"]
REFERENCE_MODEL = "grounding_dino"
OVERWRITE = False
DRY_RUN = False


# 04 — Build frozen `g_fixed` stores

This is the conceptual core of the bundle. A per-tile MLLM count is a frozen proposal function `g(s)`, just like a detector’s zero-shot count. The notebook reuses the sibling VLM store’s manifest and tiling configuration, aligns predictions by `tileid`, checks the ground truth, and writes `snapshots.npz` with one `(1, N)` `g` vector.


In [ ]:
import io
import shutil
import numpy as np
import pandas as pd
from IPython.display import display

USECOLS = ["dataset", "task", "tileid", "ground_truth_label", "final_value", "model_label", "modelstr"]

def slugify(modelstr: str) -> str:
    name = re.sub(r"[^a-z0-9.]+", "-", modelstr.rsplit("/", 1)[-1].lower()).strip("-.")
    if not name:
        raise ValueError(f"Could not create a directory slug for {modelstr!r}")
    return name

def reference_store(class_dir: Path) -> Path:
    candidates = [class_dir / f"{REFERENCE_MODEL}__g_fixed", *sorted(class_dir.glob("*__g_fixed"))]
    for candidate in candidates:
        if (candidate / "manifest.csv").is_file() and (candidate / "config.json").is_file():
            return candidate
    raise FileNotFoundError(f"No sibling g_fixed store with manifest/config under {class_dir}")

def build_g(predictions: pd.DataFrame, manifest: pd.DataFrame, class_key: str) -> np.ndarray:
    pred = predictions.sort_values("tileid").reset_index(drop=True)
    man = manifest.sort_values("tile_index").reset_index(drop=True)
    if len(pred) != len(man):
        raise ValueError(f"{class_key}: prediction rows {len(pred)} != manifest rows {len(man)}")
    if not np.array_equal(pred["tileid"].to_numpy(), man["tile_index"].to_numpy()):
        raise ValueError(f"{class_key}: tileid and tile_index do not align")
    if not np.array_equal(pred["ground_truth_label"].to_numpy(), man["gt_count"].to_numpy()):
        raise ValueError(f"{class_key}: ground truth disagrees with canonical manifest")
    values = pd.to_numeric(pred["final_value"], errors="coerce").to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError(f"{class_key}: non-finite final_value found")
    return values.astype(np.float32).reshape(1, len(values))

def write_store(store: Path, reference: Path, config: dict, g: np.ndarray) -> None:
    store.mkdir(parents=True, exist_ok=True)
    (store / "config.json").write_text(json.dumps(config, indent=2) + "\n")
    shutil.copyfile(reference / "manifest.csv", store / "manifest.csv")
    buffer = io.BytesIO()
    np.savez(buffer, g=g, snapshot_steps=np.asarray([0], dtype=np.int64))
    (store / "snapshots.npz").write_bytes(buffer.getvalue())


## Align and write stores

`DRY_RUN=True` checks every group without writing. `OVERWRITE=False` is the safer default when a store already exists. The output layout is the same one consumed by the existing simulation code.


In [ ]:
frames = []
for source in PREDICTION_CSVS:
    if source.is_file():
        frame = pd.read_csv(source, usecols=USECOLS, low_memory=False)
        frame["source_csv"] = source.name
        frames.append(frame)
if not frames:
    raise FileNotFoundError("No prediction CSVs found. Set PREDICTION_CSVS in the parameter cell.")
predictions = pd.concat(frames, ignore_index=True)
store_rows = []
for (dataset, task, modelstr), subset in predictions.groupby(["dataset", "task", "modelstr"], sort=True):
    class_key = f"{dataset}/{task}"
    class_dir = RUN_ROOT / dataset / task
    reference = reference_store(class_dir)
    manifest = pd.read_csv(reference / "manifest.csv")
    g = build_g(subset, manifest, class_key)
    labels = subset["model_label"].dropna().unique()
    if len(labels) != 1:
        raise ValueError(f"{class_key}/{modelstr}: expected one model label, found {labels}")
    slug = slugify(modelstr)
    store = class_dir / f"{slug}__g_fixed"
    reference_config = json.loads((reference / "config.json").read_text())
    config = {key: value for key, value in reference_config.items() if key not in {"score_thresh", "capture_score_floor", "text_prompt", "text_prompt_resolved"}}
    config.update({"run_id": f"{slug}__g_fixed", "regime": "g_fixed", "class_key": class_key, "model": slug, "model_label": str(labels[0]), "modelstr": modelstr, "source_csv": str(subset["source_csv"].iloc[0]), "F_model": float(g.sum())})
    action = "skip-existing" if store.exists() and not OVERWRITE else ("dry-run" if DRY_RUN else "write")
    if action == "write":
        write_store(store, reference, config, g)
    store_rows.append({"class_key": class_key, "model": str(labels[0]), "N": g.shape[1], "F_model": float(g.sum()), "action": action, "store": str(store)})
summary = pd.DataFrame(store_rows)
display(summary.head(12))
print(summary["action"].value_counts().to_dict())


## Visual checks on the frozen vectors

Totals expose outliers; the diagonal check exposes a shifted or mis-parsed count vector before simulation.


In [ ]:
from mu.visv2.figure import Figure

fig = Figure(n_rows=1, n_cols=2, figsize=(12, 4.5), fontfamily="sans-serif", fontsize=11, panel_padding=(0.3, 0.2))
left, right = fig.panels
totals = summary.groupby("model", as_index=False)["F_model"].sum().sort_values("F_model", ascending=False).head(12)
left.p.bar(totals["model"].tolist(), totals["F_model"].tolist(), color="#0891b2")
left.add_xticklabel_rotation(70)
left.set_yaxislabel("sum of frozen g(s)")
left.set_title("Frozen proposal totals")
one_task = predictions[(predictions["dataset"] + "/" + predictions["task"]) == TASKS[0]]
for model, subset in one_task.groupby("model_label"):
    right.p.scatter(subset["ground_truth_label"], subset["final_value"], s=12, alpha=0.25, label=model)
maximum = max(one_task["ground_truth_label"].max(), one_task["final_value"].max())
right.p.plot([0, maximum], [0, maximum], "--", color="#dc2626")
right.set_xaxislabel("true count")
right.set_yaxislabel("frozen g(s)")
right.set_title(f"Alignment view: {TASKS[0]}")
right.p.legend(frameon=False, fontsize=7)
fig.show()
